# MIL-CREDA frente a CREDA — fase uno: la corrida

Este cuaderno corre la campaña y nada más: el pronóstico de costo, la búsqueda del techo de cada familia y la campaña completa. No arma ninguna tabla ni conclusión — eso vive en `Benchmark_Phase1_Report.ipynb`, que lee `summary.json`, `runs.jsonl` y el registro de la búsqueda desde `MIL-CREDA/Results/Benchmark/` y nunca vuelve a entrenar nada.

Separar los dos es lo que hace barata una corrección al informe: re-renderizarlo cuesta segundos porque no toca la campaña, en vez del costo de cómputo entero.

In [ ]:
# Bootstrap: locate the repository wherever this is running, and import from it.
# Local, Colab and Kaggle differ only in where the checkout sits.
import os
import sys
from pathlib import Path


def find_repository() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (os.environ.get("MIL_CREDA_REPO", ""), "/content", "/kaggle/working"):
        if base and Path(base).is_dir():
            candidates.append(Path(base))
            candidates.extend(sorted(Path(base).glob("*")))
    for candidate in candidates:
        if (candidate / "src" / "MIL_CREDA_Benchmark").is_dir():
            return candidate.resolve()
    raise SystemExit(
        "cannot find the repository. Set MIL_CREDA_REPO to the checkout that "
        "holds src/MIL_CREDA_Benchmark, or run this notebook from inside it."
    )


REPOSITORY = find_repository()
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

In [ ]:
import json
import time

from IPython.display import Markdown, display

from MIL_CREDA_Benchmark import config, harness


def show(text: str) -> None:
    """Una tabla se muestra como tabla, no como texto de ancho fijo.

    Es la misma cadena que va al registro, renderizada. Nada se vuelve a
    calcular acá: si esto y el archivo dijeran cosas distintas, habría dos
    versiones del mismo número y ninguna forma de saber cuál se movió.
    """
    display(Markdown(text))


device = harness.resolve_device()
reduction = harness.Reduction(device=str(device), environment=harness.environment())

shape = config.sizing()
print(json.dumps(shape, indent=2))
print()
print(harness.header(reduction))
print()
print("environment:", reduction.environment["platform"],
      "| torch", reduction.environment["torch"],
      "| self-hosted" if reduction.environment["selfHosted"] else "| hosted runtime")

In [ ]:
# One run, timed, before committing to the whole grid. An estimate of the cost is
# cheaper than the cost, so it happens first.
from MIL_CREDA_Benchmark import bags

material = {role: bags.build(code, config.DATA_CACHE, config.SEEDS[0])
            for role, code in zip(("source", "target"), config.TRANSFERS[0])}
probe = harness.run_one("G", config.TRANSFERS[0], config.SEEDS[0],
                        reduction, device, material)
per_run = probe["seconds"]
full = len(config.ARMS) * len(config.TRANSFERS) * 30 * per_run * 20 / reduction.epochs
# The search's own forecast, and it goes first because the search goes first. Its
# epochs are its own — never the pilot's — so the ratio is part of the estimate.
busqueda = shape["search"]
segundos = busqueda["runs"] * per_run * busqueda["epochs"] / reduction.epochs
print(f"one full arm, {reduction.epochs} epochs: {per_run:.1f}s")
print(f"the ceiling search ({busqueda['runs']} runs at {busqueda['epochs']} epochs): "
      f"about {segundos / 60:.0f} min"
      + ("" if harness.search_record() is None else "  — already on record, skipped"))
print(f"this grid ({shape['runs']} runs): about {shape['runs'] * per_run / 60:.0f} min")
print(f"at 20 epochs and 30 seeds: about {full / 3600:.0f} h")
del material, probe

## La búsqueda del techo

Antes de comparar nada hay que elegir un escalar: hasta dónde sube la rampa del
término de adaptación. Uno solo para las dos familias iguala el coeficiente y
desiguala el balance —los dos objetivos están separados por un factor `B_src`, así
que un mismo número pone la adaptación en la mayor parte de un objetivo y en una
décima parte del otro— por eso se busca **uno por familia**, y cada derivación
hereda el de la suya.

Es un experimento y se declara como tal. Corre sobre `SEARCH_TRANSFERS` y mide
sobre las bolsas de **validación**, nunca sobre el material del que se lee el
veredicto: elegir por resultado ahí haría que el veredicto informe una decisión que
él mismo ya tomó. La elección se hace por diferencias apareadas dentro de cada
`(semilla, transferencia)`, y un empate va al techo más chico —el mismo resultado
con menos adaptación es la afirmación más débil.

Y corre a **su** escala, no a la del piloto. La rampa sube sobre la fracción de
entrenamiento transcurrida: con tres épocas satura en la segunda y todo techo se
alcanza casi enseguida, así que un techo encontrado ahí describe un paisaje en el
que la campaña no entrena nunca. Es la única parte de este cuaderno que no tiene
escala de piloto, y por eso es la más larga.

Se busca una vez. Si `ceilings.json` ya existe se lee y no se vuelve a buscar: que
el registro exista significa que la búsqueda contestó, y sobrescribir una respuesta
porque alguien quería otra es exactamente el refinanciamiento silencioso que la
campaña se niega a hacer. Para volver a empezar hay que borrarlo a mano.

In [ ]:
from dataclasses import replace

# Los techos vigentes, buscados una sola vez si todavía no hay registro. La
# `reduction` se reconstruye con lo que dice el registro y no con lo que quedó en
# memoria: `config.CEILINGS` se llena al importar, y si la búsqueda corre en este
# mismo proceso ese mapeo sigue vacío y la campaña se negaría con razón.
reduction = replace(reduction, ceilings=harness.ceilings_in_force(reduction, device))

## La corrida

Una línea por transferencia, no una por corrida: con treinta semillas la lista
completa serían mil ochocientas líneas y todo lo que dicen ya está en las tablas
de más abajo. Esta salida existe para saber que la campaña sigue viva, nada más.

In [ ]:
seen = {"n": 0}

def progress(line: str) -> None:
    seen["n"] += 1
    if seen["n"] % len(config.ARMS) == 0:
        print(f"  {seen['n']:>5}/{shape['runs']} corridas "
              f"({(time.perf_counter() - started) / 60:.1f} min)")

started = time.perf_counter()
summary = harness.campaign(reduction, device, progress=progress)
runs = [json.loads(line) for line in
        (config.RESULTS / "runs.jsonl").read_text().splitlines() if line.strip()]
print(f"\ncampaña terminada en {(time.perf_counter() - started) / 60:.1f} min, "
      f"{len(runs)} corridas")